### Mosaic Setup

In [0]:
%pip install uv

In [0]:
### https://github.com/astral-sh/uv/issues/8671
# need to use non-default virtual env location for `uv` on Databricks. will piggyback on the existing virtual env Databricks has setup automatically.
# python version specified in pyproject.toml should match Databricks Runtime python version (no known workaround).
import os
path_current = os.getcwd()
os.environ["UV_PROJECT_ENVIRONMENT"] = os.environ["VIRTUAL_ENV"]
path_mosaic = os.path.abspath("/Workspace/Users/karthik.raj@bio-techne.com/mosaic")
os.chdir(path_mosaic)

In [0]:
import os
# Must be set BEFORE `import jax` to take effect
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.95"  # Use 95% of GPU (default 75%)
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"   # Allocate on-demand, not upfront
# os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"  # Optional: use CUDA allocator (slower but frees memory)

In [0]:
%%sh
uv add jax[cuda12]
uv sync --group jax-cuda --extra torch-cuda
uv pip install .

### Motif Scaffolding Preparation

1. Creating Target Distogram based on the motif's CB distances
2. Create Motif-Specific Distogram based Loss
3. Create Linker Sampler to sample lengths between motifs

        Final Binder Length = Sum(Linker Lengths) + Sum(Motif Lengths)


In [0]:
import os
# Optional: Set the working directory explicitly to the current workspace folder
import sys
os.chdir("/Workspace/Users/karthik.raj@bio-techne.com/auto_mosaic")

print(f"Current Working Directory: {os.getcwd()}")

In [0]:
import torch

# 1. Force PyTorch to initialize the CUDA driver context immediately
torch.cuda.init() 
print(torch.__version__, torch.version.cuda)
print(torch._C._GLIBCXX_USE_CXX11_ABI)

In [0]:
%%sh
uv pip install flash-attn==2.8.3 --no-build-isolation
uv pip install databricks-sdk  

In [0]:
!python prepare.py

## Hyperparameter Grid Search (train_ml.py + MLflow)

Sweeps composite loss-function weight combinations. Each trial runs `train_ml.py` as a subprocess
(fresh GPU/JAX state per run, since `model_boltz`/`model_mpnn` are reloaded each time) with a fixed
seed shared across all configs in the sweep. Reusing the same seed across configs ("common random
numbers") means differences in the resulting metrics reflect the weight choices rather than which
config got luckier on PSSM initialization. Each trial logs its params/metrics/structures as an
MLflow run nested under one parent run for the whole sweep.

Once the broad sweep narrows down to a few promising configs, re-run just those with a handful of
different seeds (edit `SEED` below, or extend the loop) and average their composite_score before
picking a winner — a single seed isn't reliable enough to declare one config strictly better.

### MLFlow & Optuna Setup

In [0]:
import mlflow
mlflow.login()
# Create mlflow experiment
experiment_name = f"mosaic_motif_scaffolding_gopher_alpha"
full_experiment_name = f"/Users/karthik.raj@bio-techne.com/{experiment_name}"
if not mlflow.get_experiment_by_name(full_experiment_name):
    mlflow.create_experiment(full_experiment_name)
 
# Start mlflow run here to avoid multiple runs being created automatically
mlflow.set_experiment(full_experiment_name)

In [0]:
import shutil
import optuna
def databricks_optuna_storage(
    optuna_file_path: str, tmp_optuna_file_path: str
) -> optuna.storages.JournalStorage:
    """
    Create an Optuna JournalStorage object for Databricks.

    Parameters
    ----------
    optuna_file_path : str
        The path to the Optuna study file.
    tmp_optuna_file_path : str
        The temporary path to store the Optuna study file.

    Returns
    -------
    optuna.storages.JournalStorage
        The JournalStorage object for Databricks.

    Notes
    -----
    - It uses `shutil.copy` to copy the `optuna_file_path` to `tmp_optuna_file_path` if the file exists.
        - This is done because databricks dbfs storage does not allow for append operations
        - https://kb.databricks.com/dbfs/errno95-operation-not-supported
    - The `lock_obj` is created using `optuna.storages.JournalFileOpenLock` to ensure exclusive access to the file.
    - The `file_storage` is created using `optuna.storages.JournalFileStorage` with the `lock_obj`.
    - Finally, the `storage` is created using `optuna.storages.JournalStorage` with the `file_storage`.
    """
    if os.path.isfile(optuna_file_path):
        shutil.copy(optuna_file_path, tmp_optuna_file_path)

    storage = optuna.storages.RDBStorage(
        url=f"sqlite:///{tmp_optuna_file_path}",
        failed_trial_callback=optuna.storages.RetryFailedTrialCallback(max_retry=3),
    )

    return storage

In [0]:
class DatabricksOptunaStorageCallback:
    """
    A callback class for storing Optuna study progress in Databricks.

    Parameters
    ----------
    optuna_file_path : str
        The file path to store the Optuna study progress.
    tmp_optuna_file_path : str
        The temporary file path for storing the Optuna study progress.
    """

    def __init__(self, optuna_file_path: str, tmp_optuna_file_path: str):
        self.optuna_file_path = optuna_file_path
        self.tmp_optuna_file_path = tmp_optuna_file_path

    def __call__(
        self, study: optuna.study.Study, trial: optuna.trial.FrozenTrial
    ) -> None:
        """
        Callback function to store Optuna study progress.

        Parameters
        ----------
        study : optuna.study.Study
            The Optuna study object.
        trial : optuna.trial.FrozenTrial
            The Optuna frozen trial object.

        Returns
        -------
        None
        """
        try:
            shutil.copy(self.tmp_optuna_file_path, self.optuna_file_path)
            print("Optuna save activity succeeded")
        except Exception as e:
            print(f"Optuna save activity failed: {e}")

In [0]:
# Create optuna storage
optuna_file_path = f"/Workspace/Users/karthik.raj@bio-techne.com/auto_mosaic/grid_search/{experiment_name}_hpt.db"
os.makedirs(os.path.dirname(optuna_file_path), exist_ok=True)
tmp_optuna_file_path = f"/tmp/{experiment_name}_hpt.db"

if os.path.exists(tmp_optuna_file_path):
    os.remove(tmp_optuna_file_path)

storage = databricks_optuna_storage(
    optuna_file_path=optuna_file_path, tmp_optuna_file_path=tmp_optuna_file_path
)
databricks_optuna_storage_cb = DatabricksOptunaStorageCallback(
    optuna_file_path=optuna_file_path, tmp_optuna_file_path=tmp_optuna_file_path
)

In [0]:
import json
import math
import os
import subprocess
import tempfile

import mlflow
import optuna

# Fixed seed shared across every grid config (common random numbers): keeps the comparison
# between configs from being confounded by which one happened to get a luckier PSSM init.
SEED = 42

# Baseline weights -- grid points below override only the swept keys, everything else stays fixed.
BASE_WEIGHTS = {
    'binder_contact': 0.5,
    'within_binder_contact': 0.5,
    'inverse_folding_seq_recovery': 6.0,
    'target_binder_pae': 0.05,
    'binder_target_pae': 0.05,
    'within_binder_pae': 0.4,
    'iptm': 0.025,
    'ptm_energy': 0.025,
    'plddt': 0.025,
    'anti_helix': 0.1,
    'bias_sheet': 0.5,
    'first_motif_distogram': 0.1,
    'first_motif_rmsd': 3.0,
    'second_motif_distogram': 0.1,
    'second_motif_rmsd': 3.0,
}

GRID = {
    'anti_helix': [0.5],
    'bias_sheet': [0],
    'first_motif_distogram' : [3.0],
    'second_motif_distogram' : [3.0],
    'first_motif_rmsd' : [1.5],
    'second_motif_rmsd' : [1.5]
}

with mlflow.start_run(run_name="grid_search_sweep") as parent_run:
    parent_run_id = parent_run.info.run_id

    def objective(trial: optuna.Trial) -> float:
        overrides = {key: trial.suggest_categorical(key, values) for key, values in GRID.items()}
        weights = {**BASE_WEIGHTS, **overrides}
        print(f"Trial {trial.number} overrides: {overrides}")

        # Pass in Databricks Credentials
        context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        db_host = context.apiUrl().get()
        db_token = context.apiToken().get()

        with tempfile.TemporaryDirectory() as tmp_dir:
            result_path = os.path.join(tmp_dir, "trial_result.json")
            # Copy the current notebook environment variables
            current_env = os.environ.copy()
            current_env["MLFLOW_TRACKING_URI"] = "databricks"
            current_env["DATABRICKS_HOST"] = db_host
            current_env["DATABRICKS_TOKEN"] = db_token # Ensure it knows to look for Databricks
            try:
                proc = subprocess.run(
                    [
                        "python", "train_ml.py", json.dumps(weights),
                        "--seed", str(SEED),
                        "--parent_run_id", parent_run_id,
                        "--result_path", result_path,
                    ],
                    check=True,
                    capture_output= False,
                    text=True,
                    env = current_env
                )
            except subprocess.CalledProcessError as e:
                raise RuntimeError(
                    f"train_ml.py failed (exit={e.returncode}).\n"
                    f"STDOUT:\n{e.stdout}\nSTDERR:\n{e.stderr}"
                ) from e
            if not os.path.exists(result_path):
                raise RuntimeError(
                    f"train_ml.py exited 0 but did not write result file.\n"
                    f"STDOUT:\n{proc.stdout}\nSTDERR:\n{proc.stderr}"
                )
            with open(result_path) as f:
                metrics = json.load(f)

        trial.set_user_attr("metrics", metrics)
        return metrics["composite_score"]
    
    try:
        optuna.delete_study(study_name=f"{experiment_name}_hpt", storage=storage)
    except KeyError:
        pass  # study doesn't exist yet — nothing to delete

    study = optuna.create_study(
        study_name= f"{experiment_name}_hpt",
        storage= storage,
        load_if_exists=True,
        direction="minimize",
        sampler=optuna.samplers.GridSampler(search_space=GRID),
    )
    n_trials = math.prod(len(values) for values in GRID.values())
    study.optimize(objective, n_trials=n_trials, callbacks=[databricks_optuna_storage_cb])

print(f"Best composite_score: {study.best_value:.4f}")
print(f"Best weight overrides: {study.best_params}")